In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')


In [2]:
df=pd.read_csv('daily-minimum-temperatures-in-me.csv',encoding="utf-8", sep=',', on_bad_lines='skip')
df.head()

,Date,"Daily minimum temperatures in Melbourne, Australia, 1981-1990"
0,1981-01-01,20.7
1,1981-01-02,17.9
2,1981-01-03,18.8
3,1981-01-04,14.6
4,1981-01-05,15.8


In [3]:
df.columns

Index(['Date', 'Daily minimum temperatures in Melbourne, Australia, 1981-1990'], dtype='object')

In [4]:
df.rename(columns={"Daily minimum temperatures in Melbourne, Australia, 1981-1990":"Temp"},inplace=True)

In [5]:
df.head()

,Date,Temp
0,1981-01-01,20.7
1,1981-01-02,17.9
2,1981-01-03,18.8
3,1981-01-04,14.6
4,1981-01-05,15.8


In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3650 entries, 0 to 3649
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   Date    3650 non-null   object
 1   Temp    3650 non-null   object
dtypes: object(2)
memory usage: 57.2+ KB


In [7]:
df.isnull().sum()

Date    0
Temp    0
dtype: int64

In [8]:
df['Date'] = pd.to_datetime(df['Date'])
df.set_index('Date', inplace=True)

In [9]:
df['Temp']=pd.to_numeric(df['Temp'],errors='coerce')
df.dropna(inplace=True)

In [10]:
values=df["Temp"].values

In [11]:
df.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 3647 entries, 1981-01-01 to 1990-12-31
Data columns (total 1 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   Temp    3647 non-null   float64
dtypes: float64(1)
memory usage: 57.0 KB


In [12]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler

In [13]:
scaler = MinMaxScaler()
df["Temp"]=scaler.fit_transform(df[["Temp"]])


In [14]:
import numpy as np

def create_sequences(data, time_steps=30):
    X, y = [], []
    for i in range(len(data) - time_steps):
        X.append(data[i:i+time_steps])
        y.append(data[i+time_steps])
    return np.array(X), np.array(y)

X, y = create_sequences(df["Temp"], 30)

In [15]:
X_train, X_test,y_train,y_test= train_test_split(X,y, test_size=0.2,shuffle=False)

In [16]:
import tensorflow
from keras.models import Sequential
from keras.layers import GRU,Dense,Dropout
from keras.callbacks import EarlyStopping

In [17]:
model=Sequential()

In [18]:
model.add(GRU(512,return_sequences=True,activation="tanh",input_shape=(50,1)))
model.add(Dropout(0.5))
model.add(GRU(256,return_sequences=True,activation="tanh"))
model.add(Dropout(0.5))
model.add(GRU(128,return_sequences=True,activation="tanh"))
model.add(Dropout(0.5))
model.add(GRU(64,return_sequences=True,activation="tanh"))
model.add(Dropout(0.5))
model.add(GRU(32,return_sequences=False,activation="tanh"))

model.add(Dropout(0.5))
model.add(Dense(1))

In [19]:
model.compile(optimizer="adam",loss="mse")

In [20]:
model.fit(X_train,y_train,epochs=50,validation_data=(X_test,y_test),callbacks=EarlyStopping(patience=3))

Epoch 1/50
91/91 ━━━━━━━━━━━━━━━━━━━━ 73s 572ms/step - loss: 0.0276 - val_loss: 0.0107
Epoch 2/50
91/91 ━━━━━━━━━━━━━━━━━━━━ 81s 561ms/step - loss: 0.0204 - val_loss: 0.0113
Epoch 3/50
91/91 ━━━━━━━━━━━━━━━━━━━━ 82s 560ms/step - loss: 0.0191 - val_loss: 0.0099
Epoch 4/50
91/91 ━━━━━━━━━━━━━━━━━━━━ 82s 563ms/step - loss: 0.0177 - val_loss: 0.0107
Epoch 5/50
91/91 ━━━━━━━━━━━━━━━━━━━━ 85s 599ms/step - loss: 0.0163 - val_loss: 0.0096
Epoch 6/50
91/91 ━━━━━━━━━━━━━━━━━━━━ 52s 574ms/step - loss: 0.0162 - val_loss: 0.0094
Epoch 7/50
91/91 ━━━━━━━━━━━━━━━━━━━━ 82s 577ms/step - loss: 0.0137 - val_loss: 0.0090
Epoch 8/50
91/91 ━━━━━━━━━━━━━━━━━━━━ 81s 569ms/step - loss: 0.0140 - val_loss: 0.0093
Epoch 9/50
91/91 ━━━━━━━━━━━━━━━━━━━━ 83s 575ms/step - loss: 0.0129 - val_loss: 0.0075
Epoch 10/50
91/91 ━━━━━━━━━━━━━━━━━━━━ 53s 582ms/step - loss: 0.0129 - val_loss: 0.0076
Epoch 11/50
91/91 ━━━━━━━━━━━━━━━━━━━━ 79s 550ms/step - loss: 0.0126 - val_loss: 0.0087
Epoch 12/50
91/91 ━━━━━━━━━━━━━━━━━━━━ 57

In [21]:
y_pred=model.predict(X_test)

23/23 ━━━━━━━━━━━━━━━━━━━━ 24s 638ms/step


In [22]:
y_pred=scaler.inverse_transform(y_pred)
y_test_actual=scaler.inverse_transform(y_test.reshape(-1,1))

In [23]:
from sklearn.metrics import mean_squared_error
mse = mean_squared_error(y_test_actual, y_pred)
print("Mean Squared Error:", mse)

Mean Squared Error: 6.8690216977778356
